# 01. Training & Evaluation Orchestration

This notebook orchestrates the training and multi-seed evaluation of reinforcement learning algorithms and baseline inventory control policies for the platelet supply chain management problem.

### Policies Tracked:
1. **MAPPO** (Multi-Agent PPO with lateral transshipment)
2. **SA-PPO (With Transshipment)** (Single-Agent PPO on state and action space with transshipments)
3. **SA-PPO (No Transshipment)** (Single-Agent PPO without lateral transshipments)
4. **Order-Up-To Baseline** (Standard inventory heuristic policy)

In [ ]:
# Setup python path & package imports
import os
import sys
import warnings
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath(".."))
warnings.filterwarnings("ignore")

from src.configs.config import (
    TRAIN_SEEDS, EVAL_SEEDS, MODELS_DIR,
    TOTAL_EPISODES, UPDATE_EPISODES, DEVICE
)
from src.utils.seeding import set_seed
from src.env.platelet_env import PlateletSupplyChainEnv
from src.agents.mappo import MultiAgentMAPPO
from src.agents.ppo import SingleAgentPPO
from src.baselines.order_up_to import OrderUpToBaseline
from src.evaluation.evaluation import evaluate_multi_seed, create_excel_report
from src.utils.plotting import plot_combined_learning_curves

print(f"Using device: {DEVICE}")

## 1. Train & Evaluate MAPPO Policy
Trains MAPPO models on 5 independent training seeds and evaluates them across 50 common evaluation seeds.

In [ ]:
def train_mappo(train_seed):
    print(f"\n🚀 TRAINING MAPPO - SEED {train_seed}")
    set_seed(train_seed)
    env = PlateletSupplyChainEnv(enable_transshipment=True)
    agent = MultiAgentMAPPO()

    buf_s, buf_ap, buf_al, buf_lp, buf_ll, buf_r, buf_ns, buf_d, buf_m = [], [], [], [], [], [], [], [], []
    best_reward = -float('inf')
    rewards_log = []
    model_path = os.path.join(MODELS_DIR, f"mappo_seed_{train_seed}.pt")
    os.makedirs(MODELS_DIR, exist_ok=True)

    for ep in range(1, TOTAL_EPISODES + 1):
        obs = env.reset()
        done = False
        ep_reward = 0

        while not done:
            act, ap, al, lp, ll = agent.get_action(obs)
            next_obs, r, done, info = env.step(act)

            buf_s.append(obs)
            buf_ap.append(ap)
            buf_al.append(al)
            buf_lp.append(lp)
            buf_ll.append(ll)
            buf_r.append(r)
            buf_ns.append(next_obs)
            buf_d.append(1.0 if done else 0.0)
            buf_m.append(info['order_mask'])

            obs = next_obs
            ep_reward += r

        rewards_log.append(ep_reward)
        if ep_reward > best_reward:
            best_reward = ep_reward
            agent.save(model_path)

        if ep % UPDATE_EPISODES == 0:
            agent.update(buf_s, buf_ap, buf_al, buf_lp, buf_ll, buf_r, buf_ns, buf_d, buf_m)
            buf_s, buf_ap, buf_al, buf_lp, buf_ll, buf_r, buf_ns, buf_d, buf_m = [], [], [], [], [], [], [], [], []

        if ep % 200 == 0:
            print(f"Episode {ep} | Avg Reward (last 50): {np.mean(rewards_log[-50:]):.2f} | Best: {best_reward:.2f}")

    return model_path, rewards_log

# Run MAPPO pipeline
mappo_rewards = {}
mappo_models = []
for s in TRAIN_SEEDS:
    path, rewards = train_mappo(s)
    mappo_models.append((s, path))
    mappo_rewards[s] = rewards

# Save combined learning curve
plot_combined_learning_curves(mappo_rewards, prefix="mappo")

# Multi-seed evaluation
all_h1, all_h2, all_sc, raw_rows = [], [], [], []
env_eval = PlateletSupplyChainEnv(enable_transshipment=True)
for seed, path in mappo_models:
    agent = MultiAgentMAPPO()
    agent.load(path)
    h1_sum, h2_sum, sc_sum, raw = evaluate_multi_seed(env_eval, agent, train_seed=seed, is_mappo=True, eval_seeds=EVAL_SEEDS)
    all_h1.append(h1_sum)
    all_h2.append(h2_sum)
    all_sc.append(sc_sum)
    raw_rows.extend(raw)

create_excel_report(all_h1, all_h2, all_sc, raw_rows, TRAIN_SEEDS, "All Result - MAPPO.xlsx")

## 2. Train & Evaluate Single-Agent PPO with Transshipment (SA-T)
Trains SA-T models and performs multi-seed evaluation.

In [ ]:
def train_sa_t(train_seed):
    print(f"\n🚀 TRAINING SA-PPO WITH TRANSSHIPMENT - SEED {train_seed}")
    set_seed(train_seed)
    env = PlateletSupplyChainEnv(enable_transshipment=True)
    agent = SingleAgentPPO(action_dim=3)

    buf_s, buf_a, buf_lp, buf_r, buf_ns, buf_d, buf_m = [], [], [], [], [], [], []
    best_reward = -float('inf')
    rewards_log = []
    model_path = os.path.join(MODELS_DIR, f"sa_t_seed_{train_seed}.pt")

    for ep in range(1, TOTAL_EPISODES + 1):
        obs = env.reset()
        done = False
        ep_reward = 0

        while not done:
            act, lp = agent.get_action(obs)
            next_obs, r, done, info = env.step(act)

            buf_s.append(obs)
            buf_a.append(act)
            buf_lp.append(lp)
            buf_r.append(r)
            buf_ns.append(next_obs)
            buf_d.append(1.0 if done else 0.0)
            buf_m.append(info['order_mask'])

            obs = next_obs
            ep_reward += r

        rewards_log.append(ep_reward)
        if ep_reward > best_reward:
            best_reward = ep_reward
            agent.save(model_path)

        if ep % UPDATE_EPISODES == 0:
            agent.update(buf_s, buf_a, buf_lp, buf_r, buf_ns, buf_d, buf_m)
            buf_s, buf_a, buf_lp, buf_r, buf_ns, buf_d, buf_m = [], [], [], [], [], [], []

        if ep % 200 == 0:
            print(f"Episode {ep} | Avg Reward (last 50): {np.mean(rewards_log[-50:]):.2f} | Best: {best_reward:.2f}")

    return model_path, rewards_log

# Run SA-T pipeline
sa_t_rewards = {}
sa_t_models = []
for s in TRAIN_SEEDS:
    path, rewards = train_sa_t(s)
    sa_t_models.append((s, path))
    sa_t_rewards[s] = rewards

plot_combined_learning_curves(sa_t_rewards, prefix="sa-t")

# Multi-seed evaluation
all_h1, all_h2, all_sc, raw_rows = [], [], [], []
env_eval = PlateletSupplyChainEnv(enable_transshipment=True)
for seed, path in sa_t_models:
    agent = SingleAgentPPO(action_dim=3)
    agent.load(path)
    h1_sum, h2_sum, sc_sum, raw = evaluate_multi_seed(env_eval, agent, train_seed=seed, is_mappo=False, eval_seeds=EVAL_SEEDS)
    all_h1.append(h1_sum)
    all_h2.append(h2_sum)
    all_sc.append(sc_sum)
    raw_rows.extend(raw)

create_excel_report(all_h1, all_h2, all_sc, raw_rows, TRAIN_SEEDS, "All Result-SA-T.xlsx")

## 3. Train & Evaluate Single-Agent PPO without Transshipment (SA-NT)
Trains SA-NT models and performs multi-seed evaluation.

In [ ]:
def train_sa_nt(train_seed):
    print(f"\n🚀 TRAINING SA-PPO NO TRANSSHIPMENT - SEED {train_seed}")
    set_seed(train_seed)
    env = PlateletSupplyChainEnv(enable_transshipment=False)
    agent = SingleAgentPPO(action_dim=2)

    buf_s, buf_a, buf_lp, buf_r, buf_ns, buf_d, buf_m = [], [], [], [], [], [], []
    best_reward = -float('inf')
    rewards_log = []
    model_path = os.path.join(MODELS_DIR, f"sa_nt_seed_{train_seed}.pt")

    for ep in range(1, TOTAL_EPISODES + 1):
        obs = env.reset()
        done = False
        ep_reward = 0

        while not done:
            act, lp = agent.get_action(obs)
            next_obs, r, done, info = env.step(act)

            buf_s.append(obs)
            buf_a.append(act)
            buf_lp.append(lp)
            buf_r.append(r)
            buf_ns.append(next_obs)
            buf_d.append(1.0 if done else 0.0)
            buf_m.append(info['order_mask'])

            obs = next_obs
            ep_reward += r

        rewards_log.append(ep_reward)
        if ep_reward > best_reward:
            best_reward = ep_reward
            agent.save(model_path)

        if ep % UPDATE_EPISODES == 0:
            agent.update(buf_s, buf_a, buf_lp, buf_r, buf_ns, buf_d, buf_m)
            buf_s, buf_a, buf_lp, buf_r, buf_ns, buf_d, buf_m = [], [], [], [], [], [], []

        if ep % 200 == 0:
            print(f"Episode {ep} | Avg Reward (last 50): {np.mean(rewards_log[-50:]):.2f} | Best: {best_reward:.2f}")

    return model_path, rewards_log

# Run SA-NT pipeline
sa_nt_rewards = {}
sa_nt_models = []
for s in TRAIN_SEEDS:
    path, rewards = train_sa_nt(s)
    sa_nt_models.append((s, path))
    sa_nt_rewards[s] = rewards

plot_combined_learning_curves(sa_nt_rewards, prefix="sa-nt")

# Multi-seed evaluation
all_h1, all_h2, all_sc, raw_rows = [], [], [], []
env_eval = PlateletSupplyChainEnv(enable_transshipment=False)
for seed, path in sa_nt_models:
    agent = SingleAgentPPO(action_dim=2)
    agent.load(path)
    h1_sum, h2_sum, sc_sum, raw = evaluate_multi_seed(env_eval, agent, train_seed=seed, is_mappo=False, eval_seeds=EVAL_SEEDS)
    all_h1.append(h1_sum)
    all_h2.append(h2_sum)
    all_sc.append(sc_sum)
    raw_rows.extend(raw)

create_excel_report(all_h1, all_h2, all_sc, raw_rows, TRAIN_SEEDS, "All Result-SA-NT.xlsx")

## 4. Evaluate Order-Up-To Baseline (OUT)
Evaluates Order-Up-To baseline policy configurations on 50 evaluation seeds.

In [ ]:
print("\n🚀 EVALUATING ORDER-UP-TO HEURISTIC BASELINE")
percentages = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
all_h1, all_h2, all_sc, raw_rows = [], [], [], []
env_eval = PlateletSupplyChainEnv(enable_transshipment=False)

for pct in percentages:
    # S1, S2 parameters matching pct target levels of hospital capacity
    target = pct * (env_eval.max_capacity / 100.0)  # e.g., pct=50 -> 50.0
    agent = OrderUpToBaseline(S1=target, S2=target)
    h1_sum, h2_sum, sc_sum, raw = evaluate_multi_seed(env_eval, agent, train_seed=f"S{pct}", is_mappo=False, eval_seeds=EVAL_SEEDS)
    all_h1.append(h1_sum)
    all_h2.append(h2_sum)
    all_sc.append(sc_sum)
    raw_rows.extend(raw)

policy_names = [f"S{p}" for p in percentages]
create_excel_report(all_h1, all_h2, all_sc, raw_rows, policy_names, "All Result-OUT.xlsx", key_name="Policy")
print("✅ All training and evaluation runs finished successfully!")